In [4]:
"""
Apply the resolved_type decisions in entity_type_lookup.csv back onto the
triples file itself, producing a NEW triples file where source_type/
target_type are already correct. Run this after find_entity_type_ambiguities.py
and after entity_type_lookup.csv has been fixed (by hand, by
apply_manual_llm_resolution.py, or by llm_resolve_entity_types.py).

This is the hand-off point between "resolve entity type ambiguity" and
"build the Neo4j graph" -- build_kg_csvs.py reads the OUTPUT of this script
and no longer needs to know anything about the lookup table.

Only rows whose resolved_source/resolved_target name is in the lookup are
touched; every other type value is left exactly as it was (it was never
ambiguous to begin with). Two new columns record what happened:
source_type_resolution / target_type_resolution ("majority_vote", "llm",
"manual_llm_session", or "" if the name was never ambiguous).

Designed for Jupyter/Colab execution. No __main__ guard -- just set the
CONFIG values below and run the whole cell/file.
"""

import pandas as pd

# ---------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------
TRIPLES_PATH = "final_formatting_cleaned_triples.xlsx"                       # <- your original triples file
LOOKUP_PATH = "entity_type_lookup.csv"               # <- from find_entity_type_ambiguities.py
OUTPUT_TRIPLES_PATH = "triples_types_resolved.xlsx"  # <- new file; build_kg_csvs.py reads this

SOURCE_COL = "resolved_source"
TARGET_COL = "resolved_target"
SOURCE_TYPE_COL = "source_type"
TARGET_TYPE_COL = "target_type"

# ---------------------------------------------------------------
# LOAD
# ---------------------------------------------------------------
if TRIPLES_PATH.lower().endswith(".csv"):
    df = pd.read_csv(TRIPLES_PATH)
else:
    df = pd.read_excel(TRIPLES_PATH)
print(f"Loaded {len(df)} triples from {TRIPLES_PATH}")

lookup_df = pd.read_csv(LOOKUP_PATH)
print(f"Loaded {len(lookup_df)} entity type decisions from {LOOKUP_PATH}")

resolved_type_map = dict(zip(lookup_df["entity_name"], lookup_df["resolved_type"]))
resolution_source_map = dict(zip(lookup_df["entity_name"], lookup_df["resolution_source"]))

# ---------------------------------------------------------------
# APPLY
# ---------------------------------------------------------------
old_source_type = df[SOURCE_TYPE_COL].copy()
old_target_type = df[TARGET_TYPE_COL].copy()

df[SOURCE_TYPE_COL] = df[SOURCE_COL].map(resolved_type_map).fillna(df[SOURCE_TYPE_COL])
df[TARGET_TYPE_COL] = df[TARGET_COL].map(resolved_type_map).fillna(df[TARGET_TYPE_COL])

df["source_type_resolution"] = df[SOURCE_COL].map(resolution_source_map).fillna("")
df["target_type_resolution"] = df[TARGET_COL].map(resolution_source_map).fillna("")

n_source_changed = int((df[SOURCE_TYPE_COL] != old_source_type).sum())
n_target_changed = int((df[TARGET_TYPE_COL] != old_target_type).sum())
print(f"Rows with source_type changed: {n_source_changed}")
print(f"Rows with target_type changed: {n_target_changed}")

still_ambiguous = (
    pd.concat([
        df[[SOURCE_COL, SOURCE_TYPE_COL]].rename(columns={SOURCE_COL: "name", SOURCE_TYPE_COL: "type"}),
        df[[TARGET_COL, TARGET_TYPE_COL]].rename(columns={TARGET_COL: "name", TARGET_TYPE_COL: "type"}),
    ])
    .dropna()
    .groupby("name")["type"]
    .nunique()
)
still_ambiguous = still_ambiguous[still_ambiguous > 1]
if len(still_ambiguous):
    print(f"WARNING: {len(still_ambiguous)} entity names still have more than one type "
          f"after applying the lookup -- they're probably not in entity_type_lookup.csv "
          f"yet. Re-run find_entity_type_ambiguities.py against this output to check.")
else:
    print("Every entity name now maps to exactly one type. Safe to build the graph.")

# ---------------------------------------------------------------
# SAVE
# ---------------------------------------------------------------
df.to_excel(OUTPUT_TRIPLES_PATH, index=False)
print(f"\nSaved {OUTPUT_TRIPLES_PATH} -- point build_kg_csvs.py at this file.")

Loaded 10324 triples from final_formatting_cleaned_triples.xlsx
Loaded 145 entity type decisions from entity_type_lookup.csv
Rows with source_type changed: 174
Rows with target_type changed: 425
Every entity name now maps to exactly one type. Safe to build the graph.

Saved triples_types_resolved.xlsx -- point build_kg_csvs.py at this file.
